# 06 Semantic Gap Analysis

In [1]:
# ==================================================
# Notebook 06
# Semantic Gap Analysis
# ==================================================

import numpy as np
import pandas as pd

from typing import TypedDict
from typing import List
from typing import Dict

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [3]:
class ResearchState(TypedDict):

    research_goal: str

    queries_executed: List[str]

    raw_findings: List[str]

    deduplicated_findings: List[str]

    missing_topics: List[str]

    conflicts: List[str]

    critic_score: float

    draft_report: str

    final_report: str

    iteration_count: int

    status: str

    errors: List[str]

    metadata: Dict

In [4]:
state = {
    "research_goal": "Compare cloud costs against competitors",
    "queries_executed": [],
    "raw_findings": [],
    "deduplicated_findings": [
        "AWS pricing increased 8%",
        "Azure pricing remained stable",
    ],
    "missing_topics": [],
    "conflicts": [],
    "critic_score": 0.0,
    "draft_report": "",
    "final_report": "",
    "iteration_count": 0,
    "status": "",
    "errors": [],
    "metadata": {},
}

In [5]:
EXPECTED_TOPICS = ["AWS", "Azure", "Google Cloud", "Oracle Cloud", "IBM Cloud"]

In [6]:
def detect_covered_topics(findings, topics):

    findings_text = " ".join(findings).lower()

    covered = []

    for topic in topics:

        if topic.lower() in findings_text:

            covered.append(topic)

    return covered

In [7]:
covered = detect_covered_topics(state["deduplicated_findings"], EXPECTED_TOPICS)

covered

['AWS', 'Azure']

In [8]:
def detect_missing_topics(findings, expected_topics):

    covered = detect_covered_topics(findings, expected_topics)

    missing = [topic for topic in expected_topics if topic not in covered]

    return missing

In [9]:
missing = detect_missing_topics(state["deduplicated_findings"], EXPECTED_TOPICS)

missing

['Google Cloud', 'Oracle Cloud', 'IBM Cloud']

In [10]:
def semantic_topic_match(finding, topic, threshold=0.60):

    finding_emb = embedding_model.encode(finding)

    topic_emb = embedding_model.encode(topic)

    score = cosine_similarity(finding_emb.reshape(1, -1), topic_emb.reshape(1, -1))[0][
        0
    ]

    return score >= threshold

In [11]:
def generate_gap_queries(missing_topics):

    queries = []

    for topic in missing_topics:

        queries.extend(
            [
                f"{topic} pricing trends",
                f"{topic} infrastructure costs",
                f"{topic} competitor comparison",
            ]
        )

    return queries

In [12]:
generate_gap_queries(missing)

['Google Cloud pricing trends',
 'Google Cloud infrastructure costs',
 'Google Cloud competitor comparison',
 'Oracle Cloud pricing trends',
 'Oracle Cloud infrastructure costs',
 'Oracle Cloud competitor comparison',
 'IBM Cloud pricing trends',
 'IBM Cloud infrastructure costs',
 'IBM Cloud competitor comparison']

In [13]:
def semantic_gap_agent(state: ResearchState):

    print("\nSemantic Gap Agent Running...")

    missing_topics = detect_missing_topics(
        state["deduplicated_findings"], EXPECTED_TOPICS
    )

    state["missing_topics"] = missing_topics

    state["metadata"]["gap_queries"] = generate_gap_queries(missing_topics)

    state["status"] = "gap_analysis_completed"

    return state

In [14]:
updated_state = semantic_gap_agent(state)


Semantic Gap Agent Running...


In [15]:
updated_state["missing_topics"]

['Google Cloud', 'Oracle Cloud', 'IBM Cloud']

In [16]:
updated_state["metadata"]["gap_queries"]

['Google Cloud pricing trends',
 'Google Cloud infrastructure costs',
 'Google Cloud competitor comparison',
 'Oracle Cloud pricing trends',
 'Oracle Cloud infrastructure costs',
 'Oracle Cloud competitor comparison',
 'IBM Cloud pricing trends',
 'IBM Cloud infrastructure costs',
 'IBM Cloud competitor comparison']

In [17]:
def coverage_percentage(state):

    covered = len(EXPECTED_TOPICS) - len(state["missing_topics"])

    return round(covered / len(EXPECTED_TOPICS) * 100, 2)

In [18]:
coverage_percentage(updated_state)

40.0

In [19]:
def gap_severity(state):

    coverage = coverage_percentage(state)

    if coverage >= 90:

        return "Low"

    elif coverage >= 70:

        return "Medium"

    return "High"

In [20]:
gap_severity(updated_state)

'High'

In [21]:
from langgraph.graph import StateGraph, END

In [22]:
graph = StateGraph(ResearchState)

In [23]:
graph.add_node("gap_analyzer", semantic_gap_agent)

In [24]:
graph.set_entry_point("gap_analyzer")

graph.add_edge("gap_analyzer", END)

In [25]:
app = graph.compile()

In [26]:
result = app.invoke(state)


Semantic Gap Agent Running...


In [27]:
EXPECTED_TOPICS

['AWS', 'Azure', 'Google Cloud', 'Oracle Cloud', 'IBM Cloud']